# Build v1 — Full Cleaned & Labeled TALIS 2024 Teacher Dataset

This notebook turns the raw `ttgintt4.csv` into **v1**: a faithful, human-readable
cleaned dataset (278,383 teachers x 630 variables), driven entirely by the codebook.

**What v1 does:** admin-missing codes -> NaN; keep meaningful codes (`5 = I don't know`,
`6 = Logically not applicable`) as labels; categorical vars -> ordered `Categorical`;
continuous/id vars stay numeric; original TALIS names kept.

**What v1 does NOT do:** build the outcome or apply modeling choices — those live in v2.

*Source: OECD TALIS 2024 teacher public-use file. Cite: OECD (2025), TALIS 2024 Database, OECD, Paris.*

## 1. Imports & paths

In [1]:
from pathlib import Path
import re
import pandas as pd

# Absolute paths so the notebook runs no matter where it is saved.
DATA_DIR = Path("/Users/ruipinghuang/Desktop/9 Data Science/1 Data/TALIS2024_teachers_NoESE_CSV")
CODEBOOK = Path("/Users/ruipinghuang/Desktop/9 Data Science/5 Codebook/talis2024_teacher_codebook.csv")
OUT      = Path("v1_ttgintt4_labeled.parquet")          # output saved next to this notebook

data_path = DATA_DIR / "ttgintt4.csv"
print("raw data exists?", data_path.exists())
print("codebook exists?", CODEBOOK.exists())

raw data exists? True
codebook exists? True


## 2. Codebook + the two cleaning helpers

Every cleaning decision is read from the codebook (the source of truth), so the
pipeline is rule-based, not hand-coded per variable.

In [2]:
cb = pd.read_csv(CODEBOOK).set_index("variable_name")   # codebook is comma-separated

# A compiled regex. "|" means OR; re.I = ignore upper/lowercase.
# .search(text) is True if ANY of these words appears -> the code is TRUE missing.
# NOTE: "Logically not applicable" is deliberately NOT here, so it is kept.
ADMIN = re.compile(r"not administered|omitted|invalid", re.I)

def na_codes(var):
    """Codes to turn into NaN = only the admin-missing ones for this variable."""
    s = cb.loc[var, "all_value_labels"]                 # e.g. "1 = Yes | 2 = No | 8 = Not administered"
    out = set()
    if pd.isna(s):
        return out                                      # no codes listed (pure id/continuous)
    for part in str(s).split("|"):                      # split into "code = label" pieces
        m = re.match(r"\s*(\d+)\s*=\s*(.*)", part)  # group(1)=code, group(2)=label
        if m and ADMIN.search(m.group(2)):              # label means admin-missing?
            out.add(int(m.group(1)))                    # collect that code
    return out

def label_map(var):
    """Build {code: label} for a categorical var, KEEPING meaningful codes (5, 6),
       dropping the admin-missing ones (8, 9, ...)."""
    m = {}
    for col in ["valid_value_labels", "special_missing_or_skip_codes"]:
        s = cb.loc[var, col]
        if pd.isna(s):
            continue
        for part in str(s).split("|"):
            mt = re.match(r"\s*(\d+)\s*=\s*(.*?)\s*$", part)   # $ = end of piece
            if mt and not ADMIN.search(mt.group(2)):               # skip admin-missing labels
                m[int(mt.group(1))] = mt.group(2)
    return dict(sorted(m.items()))                      # sorted by code -> categories in order

# quick check
for v in ["TT4G37A", "TT4G35A", "T4SELF"]:
    print(f"{v:9s} NaN-codes={na_codes(v)}  keep-labels={label_map(v)}")

TT4G37A   NaN-codes={8, 9}  keep-labels={1: 'Yes', 2: 'No', 6: 'Logically not applicable'}
TT4G35A   NaN-codes={8, 9}  keep-labels={1: 'Strongly disagree', 2: 'Disagree', 3: 'Agree', 4: 'Strongly agree', 5: 'I don’t know'}
T4SELF    NaN-codes={9998, 9999}  keep-labels={}


## 3. Load the raw data (all 630 columns)

In [3]:
df = pd.read_csv(data_path, sep=";", low_memory=False)   # sep=";" : TALIS uses semicolons
print(df.shape)                                          # (278383, 630)

(278383, 630)


## 4. Clean + label every column

One loop over all columns: turn admin-missing into `NaN`, and convert categorical
variables into ordered `Categorical` (labels shown, codes recoverable via `.cat.codes`).

In [4]:
n_cat = n_num = 0
for col in df.columns:
    nac = na_codes(col)
    if nac:
        df[col] = df[col].where(~df[col].isin(nac))          # admin-missing -> NaN
    lm = label_map(col)
    if lm:                                                   # categorical
        df[col] = df[col].astype("Int64").map(lm)           # code -> label text
        df[col] = pd.Categorical(df[col], categories=list(lm.values()), ordered=True)
        n_cat += 1
    else:                                                    # continuous / id / weight
        n_num += 1

print(f"categorical: {n_cat} | numeric/id: {n_num}")

categorical: 458 | numeric/id: 172


## 5. Sanity checks

In [5]:
print("TT4G37A categories:", list(df["TT4G37A"].cat.categories))   # keeps 'Logically not applicable'
print("TT4G35A categories:", list(df["TT4G35A"].cat.categories))   # keeps 'I don\'t know'
print("T4SELF dtype     :", df["T4SELF"].dtype)                    # numeric scale, not labeled
print("\nTT4G35A counts:")
print(df["TT4G35A"].value_counts(dropna=False))

TT4G37A categories: ['Yes', 'No', 'Logically not applicable']
TT4G35A categories: ['Strongly disagree', 'Disagree', 'Agree', 'Strongly agree', 'I don’t know']
T4SELF dtype     : float64

TT4G35A counts:
TT4G35A
NaN                  187866
Agree                 40446
I don’t know          17196
Strongly agree        14356
Disagree              13470
Strongly disagree      5049
Name: count, dtype: int64


## 6. Save v1 (Parquet preserves the Categorical dtypes & labels)

In [6]:
df.to_parquet(OUT, index=False)
print("saved:", OUT, "| size MB:", round(OUT.stat().st_size / 1e6, 1))

saved: v1_ttgintt4_labeled.parquet | size MB: 107.6
